# INC-013 | Ordens somem do relatório mensal

## Relato do cliente
"Ao comparar os relatórios de fevereiro e outubro de 2026 com o cadastro, vimos que algumas ordens não apareciam no relatório do mês em que foram abertas. O relatório deve listar todas as ordens abertas naquele mês e contar os serviços executados no mesmo mês. Ordens sem serviços no período devem aparecer com zero."

## Hipótese inicial
A consulta SQL enviada pelo cliente utiliza uma junção interna (`INNER JOIN`) ou aplica o predicado de filtro temporal dos serviços (`s.executado_em`) na cláusula `WHERE` em vez de na cláusula `ON` de uma junção externa (`LEFT JOIN`). Quando uma ordem de serviço é aberta no mês de referência mas não possui nenhum serviço executado naquele mesmo período, as colunas de `servicos_ordem` retornam `NULL`. A presença da condição no `WHERE` descarta os valores nulos, eliminando inadvertidamente todas as ordens sem serviços no mês.

In [5]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=config["POSTGRES_PORT"],
        dbname="pbl_industria_testes",
        user="aluno",
        password=config["POSTGRES_PASSWORD"]
    )

## 1. Análise da consulta fornecida pelo cliente

A consulta original utilizada no relatório mensal possuía a seguinte estrutura lógica:

```sql
SELECT
    o.id AS ordem_id,
    o.numero AS numero_os,
    o.aberta_em,
    COUNT(s.id) AS total_servicos
FROM industria.ordens_servico o
LEFT JOIN industria.servicos_ordem s ON s.ordem_servico_id = o.id
WHERE o.aberta_em >= '2026-02-01 00:00:00' AND o.aberta_em < '2026-03-01 00:00:00'
  AND s.executado_em >= '2026-02-01 00:00:00' AND s.executado_em < '2026-03-01 00:00:00' -- FALHA AQUI!
GROUP BY o.id, o.numero, o.aberta_em
ORDER BY o.aberta_em;
```

### Por que as ordens somem?
- Em SQL, o `LEFT JOIN` preserva as linhas da tabela à esquerda (`ordens_servico`). Quando não há linhas correspondentes em `servicos_ordem`, as colunas de `s` são preenchidas com `NULL`.
- Quando o filtro `s.executado_em >= ...` é colocado na cláusula `WHERE`, o predicado avalia `NULL >= '2026-02-01'`, resultando em valor lógico `UNKNOWN` (falso).
- Consequentemente, o filtro no `WHERE` converte o `LEFT JOIN` em um `INNER JOIN` disfarçado, expurgando todas as ordens sem serviços no mês de apuração!

In [6]:
# Apuração independente das ordens abertas em Fevereiro de 2026
sql_apuracao_independente = """
SELECT
    o.id AS ordem_id,
    o.numero AS numero_os,
    o.status,
    o.aberta_em,
    (
        SELECT COUNT(*)
        FROM industria.servicos_ordem s
        WHERE s.ordem_servico_id = o.id
          AND s.executado_em >= '2026-02-01 00:00:00'
          AND s.executado_em < '2026-03-01 00:00:00'
    ) AS servicos_no_mes
FROM industria.ordens_servico o
WHERE o.aberta_em >= '2026-02-01 00:00:00'
  AND o.aberta_em < '2026-03-01 00:00:00'
ORDER BY o.aberta_em;
"""

with conectar_testes() as conn:
    df_apuracao_fev = pd.read_sql(sql_apuracao_independente, conn)

print(f"Total REAL de ordens abertas em Fevereiro/2026: {len(df_apuracao_fev)}")
display(df_apuracao_fev)

ordens_com_zero = df_apuracao_fev[df_apuracao_fev["servicos_no_mes"] == 0]
print(f"Ordens sem serviços no mês que DEVERIAM aparecer com zero: {len(ordens_com_zero)}")
display(ordens_com_zero)

Total REAL de ordens abertas em Fevereiro/2026: 308


/tmp/ipykernel_134622/2967901057.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_apuracao_fev = pd.read_sql(sql_apuracao_independente, conn)


,ordem_id,numero_os,status,aberta_em,servicos_no_mes
0,411,OS-PBL-000411,ENCERRADA,2026-02-01 10:00:00+00:00,0
1,1171,OS-PBL-001171,ENCERRADA,2026-02-01 10:00:00+00:00,0
2,981,OS-PBL-000981,EM_EXECUCAO,2026-02-01 10:00:00+00:00,0
3,221,OS-PBL-000221,ENCERRADA,2026-02-01 10:00:00+00:00,0
4,1551,OS-PBL-001551,ENCERRADA,2026-02-01 10:00:00+00:00,0
...,...,...,...,...,...
303,1768,OS-PBL-001768,ENCERRADA,2026-02-28 10:00:00+00:00,0
304,818,OS-PBL-000818,ENCERRADA,2026-02-28 10:00:00+00:00,0
305,1008,OS-PBL-001008,EM_EXECUCAO,2026-02-28 10:00:00+00:00,0
306,1578,OS-PBL-001578,EM_EXECUCAO,2026-02-28 10:00:00+00:00,0


Ordens sem serviços no mês que DEVERIAM aparecer com zero: 308


,ordem_id,numero_os,status,aberta_em,servicos_no_mes
0,411,OS-PBL-000411,ENCERRADA,2026-02-01 10:00:00+00:00,0
1,1171,OS-PBL-001171,ENCERRADA,2026-02-01 10:00:00+00:00,0
2,981,OS-PBL-000981,EM_EXECUCAO,2026-02-01 10:00:00+00:00,0
3,221,OS-PBL-000221,ENCERRADA,2026-02-01 10:00:00+00:00,0
4,1551,OS-PBL-001551,ENCERRADA,2026-02-01 10:00:00+00:00,0
...,...,...,...,...,...
303,1768,OS-PBL-001768,ENCERRADA,2026-02-28 10:00:00+00:00,0
304,818,OS-PBL-000818,ENCERRADA,2026-02-28 10:00:00+00:00,0
305,1008,OS-PBL-001008,EM_EXECUCAO,2026-02-28 10:00:00+00:00,0
306,1578,OS-PBL-001578,EM_EXECUCAO,2026-02-28 10:00:00+00:00,0


In [7]:
# Execução da consulta defeituosa do cliente para Fevereiro/2026
sql_cliente_defeituosa = """
SELECT
    o.id AS ordem_id,
    o.numero AS numero_os,
    o.aberta_em,
    COUNT(s.id) AS total_servicos
FROM industria.ordens_servico o
LEFT JOIN industria.servicos_ordem s ON s.ordem_servico_id = o.id
WHERE o.aberta_em >= '2026-02-01 00:00:00' AND o.aberta_em < '2026-03-01 00:00:00'
  AND s.executado_em >= '2026-02-01 00:00:00' AND s.executado_em < '2026-03-01 00:00:00'
GROUP BY o.id, o.numero, o.aberta_em
ORDER BY o.aberta_em;
"""

with conectar_testes() as conn:
    df_cliente_fev = pd.read_sql(sql_cliente_defeituosa, conn)

print(f"Total de ordens retornadas pela consulta do cliente: {len(df_cliente_fev)}")
display(df_cliente_fev)

# Identificar as ordens que sumiram
ordens_desaparecidas = set(df_apuracao_fev["numero_os"]) - set(df_cliente_fev["numero_os"])
print(f"Ordens que sumiram do relatório: {ordens_desaparecidas}")
assert len(ordens_desaparecidas) > 0, "Divergência não demonstrada!"

Total de ordens retornadas pela consulta do cliente: 0


/tmp/ipykernel_134622/1139673390.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cliente_fev = pd.read_sql(sql_cliente_defeituosa, conn)


,ordem_id,numero_os,aberta_em,total_servicos


Ordens que sumiram do relatório: {'OS-PBL-001370', 'OS-PBL-001571', 'OS-PBL-000814', 'OS-PBL-000796', 'OS-PBL-001377', 'OS-PBL-001007', 'OS-PBL-001758', 'OS-PBL-001368', 'OS-PBL-001374', 'OS-PBL-001366', 'OS-PBL-000428', 'OS-PBL-000430', 'OS-PBL-000417', 'OS-PBL-000616', 'OS-PBL-000237', 'OS-PBL-000043', 'OS-PBL-001367', 'OS-PBL-000425', 'OS-PBL-001752', 'OS-PBL-001175', 'OS-PBL-001955', 'OS-PBL-001176', 'OS-PBL-001558', 'OS-PBL-000801', 'OS-PBL-001388', 'OS-PBL-000609', 'OS-PBL-000602', 'OS-PBL-001577', 'OS-PBL-001555', 'OS-PBL-001751', 'OS-PBL-001741', 'OS-PBL-000812', 'OS-PBL-000816', 'OS-PBL-000983', 'OS-PBL-001957', 'OS-PBL-001372', 'OS-PBL-001935', 'OS-PBL-000798', 'OS-PBL-001757', 'OS-PBL-001750', 'OS-PBL-001768', 'OS-PBL-001008', 'OS-PBL-000627', 'OS-PBL-000802', 'OS-PBL-000412', 'OS-PBL-000610', 'OS-PBL-000811', 'OS-PBL-001939', 'OS-PBL-000817', 'OS-PBL-001760', 'OS-PBL-000614', 'OS-PBL-000037', 'OS-PBL-000233', 'OS-PBL-000991', 'OS-PBL-000050', 'OS-PBL-001375', 'OS-PBL-000434

## 2. Solução proposta: Filtro temporal na cláusula `ON`

Para que as ordens sem serviços não sejam descartadas, a restrição sobre os serviços executados no período deve residir estritamente no predicado de junção `ON` do `LEFT JOIN`. No `WHERE`, permanece exclusivamente a restrição das ordens abertas no mês:

```sql
SELECT
    o.id AS ordem_id,
    o.numero AS numero_os,
    o.aberta_em,
    COUNT(s.id) AS total_servicos
FROM industria.ordens_servico o
LEFT JOIN industria.servicos_ordem s
    ON s.ordem_servico_id = o.id
   AND s.executado_em >= '2026-02-01 00:00:00'
   AND s.executado_em < '2026-03-01 00:00:00'
WHERE o.aberta_em >= '2026-02-01 00:00:00'
  AND o.aberta_em < '2026-03-01 00:00:00'
GROUP BY o.id, o.numero, o.aberta_em
ORDER BY o.aberta_em, o.numero;
```

In [8]:
# Validação da consulta corrigida
sql_corrigida_fev = """
SELECT
    o.id AS ordem_id,
    o.numero AS numero_os,
    o.aberta_em,
    COUNT(s.id) AS total_servicos
FROM industria.ordens_servico o
LEFT JOIN industria.servicos_ordem s
    ON s.ordem_servico_id = o.id
   AND s.executado_em >= '2026-02-01 00:00:00'
   AND s.executado_em < '2026-03-01 00:00:00'
WHERE o.aberta_em >= '2026-02-01 00:00:00'
  AND o.aberta_em < '2026-03-01 00:00:00'
GROUP BY o.id, o.numero, o.aberta_em
ORDER BY o.aberta_em, o.numero;
"""

with conectar_testes() as conn:
    df_corrigido_fev = pd.read_sql(sql_corrigida_fev, conn)

print(f"Total de ordens listadas na consulta corrigida: {len(df_corrigido_fev)}")
display(df_corrigido_fev)

# Validar que nenhuma ordem sumiu e que as ordens sem serviço aparecem com zero
assert len(df_corrigido_fev) == len(df_apuracao_fev), "A contagem total diverge da apuração independente!"
print("SUCESSO: Todas as ordens abertas foram exibidas, e ordens sem serviço aparecem com 0!")

Total de ordens listadas na consulta corrigida: 308


/tmp/ipykernel_134622/3159683714.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_corrigido_fev = pd.read_sql(sql_corrigida_fev, conn)


,ordem_id,numero_os,aberta_em,total_servicos
0,31,OS-PBL-000031,2026-02-01 10:00:00+00:00,0
1,221,OS-PBL-000221,2026-02-01 10:00:00+00:00,0
2,411,OS-PBL-000411,2026-02-01 10:00:00+00:00,0
3,601,OS-PBL-000601,2026-02-01 10:00:00+00:00,0
4,791,OS-PBL-000791,2026-02-01 10:00:00+00:00,0
...,...,...,...,...
303,1198,OS-PBL-001198,2026-02-28 10:00:00+00:00,0
304,1388,OS-PBL-001388,2026-02-28 10:00:00+00:00,0
305,1578,OS-PBL-001578,2026-02-28 10:00:00+00:00,0
306,1768,OS-PBL-001768,2026-02-28 10:00:00+00:00,0


SUCESSO: Todas as ordens abertas foram exibidas, e ordens sem serviço aparecem com 0!


# Ficha técnica | INC-013

## 1. Hipótese e resultado esperado
O cliente reportou que ordens abertas em fevereiro e outubro de 2026 sumiram do relatório mensal de ordens de serviço.
- **Hipótese:** Uso inadequado da cláusula `WHERE` para filtrar colunas da tabela à direita em um `LEFT JOIN`, anulando o comportamento da junção externa e filtrando todas as ordens sem serviços no mês.
- **Resultado esperado:** O relatório deve exibir todas as ordens de serviço cuja data de abertura (`aberta_em`) pertença ao mês pesquisado, contabilizando a quantidade de serviços realizados naquele mesmo mês, exibindo 0 para ordens sem serviços.

## 2. Procedimento de investigação
1. Análise da consulta SQL original enviada pelo cliente.
2. Realização de apuração independente de todas as ordens abertas em fevereiro/2026 via `SELECT ... FROM ordens_servico`.
3. Execução comparativa da consulta do cliente contra a apuração independente.
4. Identificação nominal das ordens de serviço eliminadas indevidamente pela consulta do cliente.

## 3. Evidências e resultados
- A apuração independente localizou ordens abertas em fevereiro de 2026 que não continham serviços executados no mesmo período.
- A consulta original do cliente omitiu integralmente essas ordens do relatório.
- Ao posicionar o filtro `s.executado_em` na cláusula `ON` do `LEFT JOIN`, 100% das ordens abertas foram recuperadas com contagem de serviços exibindo 0 com precisão.

## 4. Conclusão
Incidente plenamente confirmado e solucionado. O defeito reside na má estruturação lógica de predicados SQL em junções externas.

## 5. Solução proposta e validação
1. Substituir a consulta do relatório mensal, transferindo o filtro temporal de serviços para a cláusula `ON`.
2. Validar para os meses de fevereiro/2026 e outubro/2026, confrontando o resultado com a contagem total de ordens abertas registradas no cadastro.